# California Insurance Code (official) → `data/california/ins_codes/`

FindLaw is hard to scrape (Cloudflare, JS). **This notebook uses California’s official site** instead:

- **Source:** [California Legislative Information — Insurance Code (INS)](https://leginfo.legislature.ca.gov/faces/codesTOCSelected.xhtml?tocCode=INS&tocTitle=Insurance+Code)
- **Method:** `httpx` + **BeautifulSoup** (no browser). Crawl **branch TOC pages**, collect `submitCodesValues('…')` section keys, then fetch each **`codes_displaySection`** page and save **Markdown** (`.md`). Your RAG pipeline already indexes `.md`.

**Output:** `data/california/ins_codes/*.md` — one file per section (e.g. `INS_sec_100.md`).

**Coverage:** Leginfo serves much of the TOC via **JavaScript**. This notebook only sees **section links that appear in the raw HTML** of each branch page (often **dozens to a few hundred** sections, not the entire Insurance Code). To add missing sections, put one number per line in **`_extras_sections.txt`** (same folder) — e.g. `790.03` or `1871.7` — then re-run the download cell.

**SSL:** If downloads fail with certificate errors on macOS, set `VERIFY_SSL = False` in the config cell (less secure).

**Politeness:** `leginfo` `robots.txt` suggests **Crawl-Delay: 10** for bots; this notebook defaults to **0.35s** between requests — increase if you prefer to be stricter.

## 1) Dependencies

In [6]:
%pip install -q httpx beautifulsoup4 lxml certifi

You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 2) Configuration

In [7]:
from __future__ import annotations

import html as html_module
import re
import time
from pathlib import Path
from urllib.parse import quote, urljoin

import certifi
import httpx
from bs4 import BeautifulSoup

ORIGIN = "https://leginfo.legislature.ca.gov"
EXPAND_PATH = "/faces/codedisplayexpand.xhtml?tocCode=INS"
EXPAND_URL = ORIGIN + EXPAND_PATH

OUT_DIR = Path("data") / "california" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "RAG-CA-INS-Leginfo/1.0 (public codes; educational indexing)"
REQUEST_DELAY_SEC = 0.35
TIMEOUT = 60.0
MAX_BRANCH_PAGES = 2500

# Set False only if you see SSL certificate errors on your Mac
VERIFY_SSL = True

_verify = certifi.where() if VERIFY_SSL else False

## 3) HTTP client + helpers

In [8]:
BRANCH_HREF = re.compile(
    r'href="(/faces/codes_displayexpandedbranch\.xhtml[^"]+)"',
    re.I,
)
SUBMIT_CODES = re.compile(r"submitCodesValues\s*\(\s*'([^']+)'\s*,", re.I)


def is_law_section_key(s: str) -> bool:
    """First arg to submitCodesValues: keep law section numbers, drop tree keys like 2.1.1."""
    s = s.strip()
    if not s or not s[0].isdigit():
        return False
    core = s.rstrip(".")
    parts = core.split(".")
    if not parts or not all(p.isdigit() for p in parts):
        return False
    # e.g. 2.1.1 (three single-digit parts) is a TOC tree id, not INS § 2
    if len(parts) == 3 and all(len(p) == 1 for p in parts):
        return False
    return True


def extract_branch_paths(page_html: str) -> set[str]:
    out: set[str] = set()
    for m in BRANCH_HREF.finditer(page_html):
        out.add(html_module.unescape(m.group(1)))
    return out


def extract_section_keys(page_html: str) -> set[str]:
    keys: set[str] = set()
    for m in SUBMIT_CODES.finditer(page_html):
        k = m.group(1).strip()
        if is_law_section_key(k):
            keys.add(k)
    return keys


def section_sort_key(s: str) -> tuple[int, ...]:
    parts: list[int] = []
    for p in s.strip().rstrip(".").split("."):
        if p.isdigit():
            parts.append(int(p))
    return tuple(parts) if parts else (0,)


def fetch_text(client: httpx.Client, url: str) -> str:
    r = client.get(url, follow_redirects=True)
    r.raise_for_status()
    time.sleep(REQUEST_DELAY_SEC)
    return r.text


def section_display_url(section_key: str) -> str:
    # Keys from HTML look like "100." or "108.1."; sectionNum accepts both
    sn = section_key.strip()
    return f"{ORIGIN}/faces/codes_displaySection.xhtml?lawCode=INS&sectionNum={quote(sn, safe='.')}"

## 4) Discover branch pages (BFS) + all section keys

In [9]:
def discover_branches_and_sections() -> tuple[list[str], set[str]]:
    """Return (ordered_branch_paths, all_section_keys)."""
    seen_branches: set[str] = set()
    queue: list[str] = []
    all_sections: set[str] = set()

    with httpx.Client(
        headers={"User-Agent": USER_AGENT, "Accept": "text/html,*/*;q=0.8"},
        timeout=TIMEOUT,
        verify=_verify,
        http2=False,
    ) as client:
        seed_html = fetch_text(client, EXPAND_URL)
        for p in extract_branch_paths(seed_html):
            if p not in seen_branches:
                seen_branches.add(p)
                queue.append(p)
        all_sections |= extract_section_keys(seed_html)

        opened = 0
        while queue and opened < MAX_BRANCH_PAGES:
            path = queue.pop(0)
            url = urljoin(ORIGIN + "/", path.lstrip("/"))
            opened += 1
            print(f"branch [{opened}] {url}", flush=True)
            html = fetch_text(client, url)
            all_sections |= extract_section_keys(html)
            for p in extract_branch_paths(html):
                if p not in seen_branches:
                    seen_branches.add(p)
                    queue.append(p)

    ordered = sorted(seen_branches)
    return ordered, all_sections


branch_list, section_keys = discover_branches_and_sections()
print(f"Branches visited/found: {len(branch_list)}")
print(f"Unique section keys (from HTML): {len(section_keys)}")
(OUT_DIR / "_branches.txt").write_text("\n".join(branch_list), encoding="utf-8")

extras_path = OUT_DIR / "_extras_sections.txt"
if not extras_path.exists():
    extras_path.write_text(
        "# Optional: one INS section number per line (no trailing dot required).\n"
        "# Uncomment or add lines, then re-run the download cell.\n"
        "# 790.03\n# 1871.7\n",
        encoding="utf-8",
    )
for raw in extras_path.read_text(encoding="utf-8").splitlines():
    line = raw.strip()
    if not line or line.startswith("#"):
        continue
    line = line.split("#", 1)[0].strip()
    if not line:
        continue
    if not line.endswith("."):
        line = line + "."
    if is_law_section_key(line):
        section_keys.add(line)

print(f"After extras file: {len(section_keys)} section keys")
(OUT_DIR / "_section_keys.txt").write_text(
    "\n".join(sorted(section_keys, key=section_sort_key)),
    encoding="utf-8",
)

branch [1] https://leginfo.legislature.ca.gov/faces/codes_displayexpandedbranch.xhtml?tocCode=INS&division=2.&title=&part=2.&chapter=1.&article=
branch [2] https://leginfo.legislature.ca.gov/faces/codes_displayexpandedbranch.xhtml?tocCode=INS&division=2.&title=&part=2.&chapter=4.&article=
branch [3] https://leginfo.legislature.ca.gov/faces/codes_displayexpandedbranch.xhtml?tocCode=INS&division=1.&title=&part=2.&chapter=5.&article=
branch [4] https://leginfo.legislature.ca.gov/faces/codes_displayexpandedbranch.xhtml?tocCode=INS&division=2.&title=&part=2.&chapter=9.&article=
branch [5] https://leginfo.legislature.ca.gov/faces/codes_displayexpandedbranch.xhtml?tocCode=INS&division=5.&title=&part=&chapter=3.&article=
branch [6] https://leginfo.legislature.ca.gov/faces/codes_displayexpandedbranch.xhtml?tocCode=INS&division=2.&title=&part=2.&chapter=8.01.&article=
branch [7] https://leginfo.legislature.ca.gov/faces/codes_displayexpandedbranch.xhtml?tocCode=INS&division=1.&title=&part=1.&chap

276

## 5) Download each section → Markdown

Skips files that already exist (delete a `.md` to refresh one section).

In [10]:
def section_key_to_filename(key: str) -> str:
    safe = re.sub(r"[^0-9.]+", "_", key.strip().rstrip(".")).strip("_") or "unknown"
    return f"INS_sec_{safe}.md"


def extract_section_body(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    node = soup.find(id="single_law_section")
    if not node:
        return BeautifulSoup(html, "lxml").get_text("\n", strip=True)[:50_000]
    return node.get_text("\n", strip=True)


def download_sections_md(section_keys: set[str]) -> None:
    keys = sorted(section_keys, key=section_sort_key)
    ok, skip, fail = 0, 0, 0
    with httpx.Client(
        headers={"User-Agent": USER_AGENT, "Accept": "text/html,*/*;q=0.8"},
        timeout=TIMEOUT,
        verify=_verify,
        http2=False,
    ) as client:
        for i, key in enumerate(keys, 1):
            dest = OUT_DIR / section_key_to_filename(key)
            if dest.exists() and dest.stat().st_size > 50:
                skip += 1
                continue
            url = section_display_url(key)
            print(f"[{i}/{len(keys)}] {key} → {dest.name}", flush=True)
            try:
                html = fetch_text(client, url)
                body = extract_section_body(html)
                title = f"California Insurance Code (INS) — Section {key.rstrip('.')}"
                md = (
                    f"# {title}\n\n"
                    f"**Official source:** {url}\n\n"
                    f"---\n\n"
                    f"{body}\n"
                )
                dest.write_text(md, encoding="utf-8")
                ok += 1
            except Exception as e:
                print(f"  FAIL {key}: {e}", flush=True)
                fail += 1
    print(f"Done. wrote={ok} skipped_existing={skip} failed={fail}")


download_sections_md(section_keys)

Done. wrote=0 skipped_existing=62 failed=0


## 6) Re-index your RAG

Run **`python3 -m app.ingest`** (or **Re-index PDFs** in the UI). `data/california/ins_codes/*.md` is under `data/`, so it is picked up by default.